# Fruit Quality, Ripeness & Defect Assessment - Training Notebook
Runs the project on Google Colab (Runtime > Change runtime type > **GPU**).

**Before you start:** upload the whole `fruit-quality-assessment` folder (with your `dataset/` inside it) to Google Drive: `MyDrive/fruit-quality-assessment`.

## 1. Mount Drive and move into the project folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/fruit-quality-assessment

## 2. Install extra packages (Colab already has TensorFlow)

In [ ]:
!pip install -q scikit-learn pandas matplotlib opencv-python-headless Pillow

## 3. Check TensorFlow and GPU

In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

## 4. Validate the dataset (structure, corrupt files, data leakage)
Fix any error before training.

In [ ]:
for task in ['fruit', 'ripeness', 'defect']:
    print('=' * 20, task, '=' * 20)
    !python -m src.prepare_dataset check --task {task}

## 5. Train Model 1 - Fruit type

In [ ]:
!python -m src.train_fruit

## 6. Train Model 2 - Ripeness

In [ ]:
!python -m src.train_ripeness

## 7. Train Model 3 - Defect

In [ ]:
!python -m src.train_defect

## 8. View the graphs and confusion matrices (actual results from YOUR training run)

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('results/*.png')):
    print(f)
    display(Image(filename=f, width=500))

## 9. Metrics summary

In [ ]:
import json, glob, pandas as pd
rows = [json.load(open(f)) for f in sorted(glob.glob('results/*_metrics.json'))]
pd.DataFrame(rows)

## 10. Quick prediction test
Upload one image, then run the cell.

In [ ]:
from google.colab import files
from src.prediction import load_models, decode_image, predict_image
from src.quality_rules import assess_quality

uploaded = files.upload()
name, data = next(iter(uploaded.items()))
models = load_models()
results = predict_image(decode_image(data, name), models)
for task, r in results.items():
    print(f"{task:<9}: {r['label']} ({r['confidence']*100:.1f}%)")
print(assess_quality(results['defect']['label'], results['ripeness']['label']))

The trained models are in `models/`. Download that folder to your laptop, then run `streamlit run app.py`.